# Lab 3: Regulatory Document RAG
**Goal:** answer questions about real financial documents, with citations, using ChromaDB and an LLM of your choice.

**Why:** an LLM alone can invent regulatory details. RAG (Retrieval-Augmented Generation) first *finds* the relevant passages in your documents, then asks the model to answer only from them. This is Module 3, sections 4-6.

## Exercise 0: Choose your provider
**Why we do this:** the rest of the lab calls one function, `ask_llm()`. You pick the provider once here and nothing else changes.

**What the code does:** it builds `ask_llm(system, user)` for Ollama, Groq or Anthropic. Ollama and Groq both speak the OpenAI-style API, so they share code.

**Important:** Ollama runs on *your* computer. Google Colab cannot reach it. If you pick Ollama, run this notebook locally (Jupyter/VS Code). Use Groq or Anthropic on Colab.

Model names change over time. If one fails, check the provider's model list and update `MODEL`.

In [ ]:
%pip install -q chromadb pypdf openai anthropic

In [ ]:
import os, getpass

PROVIDER = "groq"   # "ollama" | "groq" | "anthropic"

DEFAULTS = {
    "ollama":    {"model": "llama3.1:8b",              "base_url": "http://localhost:11434/v1"},
    "groq":      {"model": "llama-3.3-70b-versatile",  "base_url": "https://api.groq.com/openai/v1"},
    "anthropic": {"model": "claude-sonnet-5-5",        "base_url": None},
}
MODEL = DEFAULTS[PROVIDER]["model"]

if PROVIDER == "groq" and not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
if PROVIDER == "anthropic" and not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

def ask_llm(system: str, user: str) -> str:
    if PROVIDER == "anthropic":
        import anthropic
        r = anthropic.Anthropic().messages.create(
            model=MODEL, max_tokens=800, temperature=0, system=system,
            messages=[{"role": "user", "content": user}])
        return r.content[0].text
    from openai import OpenAI
    key = os.getenv("GROQ_API_KEY") if PROVIDER == "groq" else "ollama"
    client = OpenAI(base_url=DEFAULTS[PROVIDER]["base_url"], api_key=key)
    r = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return r.choices[0].message.content

print(ask_llm("Be brief.", "Say hello in five words."))

**Expected output:** a short greeting. If you see an authentication or connection error, re-check your key (Groq/Anthropic) or that `ollama serve` is running and the model is pulled (Ollama).

## Exercise 1: Load the documents
**Why we do this:** RAG quality starts with clean text. We keep the *page number* with every piece of text so answers can cite it later.

**What the code does:** `load_pdf` reads each page with `pypdf` and returns a list of `(page_number, text)`.

**Key terms:** *Corpus* = your collection of documents.

**Documents:**
- SEBI Board memo on responsibility for AI tools (downloaded for you below). It is a proposal paper, not a final circular.
- RBI Master Direction on IT Governance (2023): download from https://www.rbi.org.in/Scripts/BS_ViewMasDirections.aspx?id=12562
- HDFC Bank Integrated Annual Report 2024-25 (large; optional): https://www.hdfcbank.com/personal/about-us/investor-relations/annual-reports

Put the PDFs in a `data/` folder next to this notebook (on Colab, use the Files panel to upload). RBI's site blocks scripted downloads, so download that file in your browser.

In [ ]:
import os, urllib.request
os.makedirs("data", exist_ok=True)
SEBI_URL = "https://www.sebi.gov.in/sebi_data/meetingfiles/dec-2024/1735042007618_1.pdf"
if not os.path.exists("data/sebi_ai_memo.pdf"):
    req = urllib.request.Request(SEBI_URL, headers={"User-Agent": "Mozilla/5.0"})
    open("data/sebi_ai_memo.pdf", "wb").write(urllib.request.urlopen(req).read())

from pypdf import PdfReader
def load_pdf(path):
    return [(i + 1, (p.extract_text() or "")) for i, p in enumerate(PdfReader(path).pages)]

DOCS = {f: load_pdf(f"data/{f}") for f in os.listdir("data") if f.lower().endswith(".pdf")}
for name, pages in DOCS.items():
    print(name, "-", len(pages), "pages,", sum(len(t) for _, t in pages), "characters")

**Expected output:** one line per PDF with its page and character counts. The SEBI memo has 6 pages. If a PDF shows near-zero characters it is a scanned image and needs OCR (a stretch goal).

**What to notice:** real PDFs contain headers, page numbers and odd line breaks. That noise ends up in your chunks.

## Exercise 2: Chunking, two ways
**Why we do this:** a model cannot read a whole document per question, and one embedding of a whole document is too vague. We split text into *chunks*. Chunk strategy strongly affects retrieval quality (Module 3, section 5).

**Key terms:**
- *Fixed-size chunking:* cut every N characters, with an *overlap* so a sentence cut in half still appears whole in the next chunk.
- *Sentence chunking:* group whole sentences up to a size limit, so no sentence is cut.

**Your task:** complete `chunk_sentences`. Hint: keep adding sentences to the current chunk until adding another would exceed `size`, then save it and start a new one.

In [ ]:
import re

def chunk_fixed(pages, doc, size=800, overlap=150):
    out = []
    for page, text in pages:
        text = re.sub(r"\s+", " ", text).strip()
        step = size - overlap
        for start in range(0, max(len(text), 1), step):
            piece = text[start:start + size]
            if piece.strip():
                out.append({"text": piece, "doc": doc, "page": page})
    return out

def chunk_sentences(pages, doc, size=800):
    out = []
    for page, text in pages:
        text = re.sub(r"\s+", " ", text).strip()
        sentences = re.split(r"(?<=[.!?])\s+", text)
        current = ""
        for s in sentences:
            # YOUR CODE HERE: if current + s fits in `size`, append it to current;
            # otherwise save `current` to `out` and start a new chunk with s.
            pass
        # YOUR CODE HERE: save the last `current` chunk if it is not empty
    return out

<details><summary>Solution (try first)</summary>

```python
for s in sentences:
    if len(current) + len(s) + 1 <= size:
        current = (current + " " + s).strip()
    else:
        if current:
            out.append({"text": current, "doc": doc, "page": page})
        current = s
if current:
    out.append({"text": current, "doc": doc, "page": page})
```
</details>

In [ ]:
fixed = [c for d, p in DOCS.items() for c in chunk_fixed(p, d)]
sent  = [c for d, p in DOCS.items() for c in chunk_sentences(p, d)]
print("fixed chunks:", len(fixed), "| sentence chunks:", len(sent))
print("\nSample fixed chunk:\n", fixed[3]["text"][:300])
print("\nSample sentence chunk:\n", sent[3]["text"][:300] if len(sent) > 3 else "(complete the exercise)")

**What to notice:** fixed chunks often begin or end mid-sentence. Sentence chunks read cleanly but vary in length.

## Exercise 3: Embed and store in ChromaDB
**Why we do this:** an *embedding* turns text into a vector so that similar meanings sit close together. Chroma stores the vectors and finds the nearest ones to a question.

**What the code does:** creates two collections, one per chunking strategy, so we can compare them. Chroma's built-in local embedding model is used, so no API key is needed (it downloads a small model on first use).

**Finance link:** the data stays in memory on your machine. For real client documents this local option matters for confidentiality.

In [ ]:
import chromadb
client = chromadb.Client()

def build(name, chunks):
    col = client.get_or_create_collection(name)
    col.add(
        ids=[f"{name}-{i}" for i in range(len(chunks))],
        documents=[c["text"] for c in chunks],
        metadatas=[{"doc": c["doc"], "page": c["page"]} for c in chunks],
    )
    return col

col_fixed = build("fixed", fixed)
col_sent  = build("sentence", sent) if sent else None
print("stored:", col_fixed.count(), "fixed chunks")

## Exercise 4: Retrieve and answer with citations
**Why we do this:** this is the "R" and "G" of RAG. We retrieve the top-k chunks, put them in the prompt, and instruct the model to answer only from them and cite the source.

**Key terms:** *top-k* = how many chunks to retrieve. *Grounding* = forcing the answer to rely on supplied text.

**What the code does:** `retrieve` queries Chroma. `answer` builds a prompt with numbered sources and tells the model to say "not found" if the sources do not contain the answer.

**Finance link:** the "not found" rule and citations are what make an answer auditable (Module 2, section 7).

In [ ]:
def retrieve(col, question, k=4):
    r = col.query(query_texts=[question], n_results=k)
    return [{"text": t, **m} for t, m in zip(r["documents"][0], r["metadatas"][0])]

SYSTEM = (
    "You answer questions about financial regulation using ONLY the numbered sources. "
    "Cite sources like [1]. If the sources do not contain the answer, reply exactly: "
    "'Not found in the provided documents.'"
)

def answer(col, question, k=4):
    hits = retrieve(col, question, k)
    sources = "\n\n".join(f"[{i+1}] ({h['doc']}, p.{h['page']})\n{h['text']}" for i, h in enumerate(hits))
    reply = ask_llm(SYSTEM, f"Sources:\n{sources}\n\nQuestion: {question}")
    return reply, hits

q = "Who is responsible for the output of AI tools used by SEBI-regulated entities?"
reply, hits = answer(col_fixed, q)
print(reply)
print("\nRetrieved from:", [(h['doc'], h['page']) for h in hits])

**Expected output:** an answer citing `[1]`, `[2]` etc., stating that regulated persons stay responsible. Wording varies by model. Check each citation against the printed page numbers.

**What to notice:** ask something the documents do *not* cover (e.g. "What is the repo rate?"). A well-grounded system says "Not found".

## Exercise 5: Compare chunking strategies
**Why we do this:** you will measure, not guess, which chunking retrieves better.

**What the code does:** for each test question you list a *keyword that must appear* in a good chunk. `hit_rate` is the share of questions where at least one of the top-k chunks contains the keyword. This is a crude proxy for *recall@k* (Module 3, section 6).

**Your task:** write 5 questions about your documents with a distinctive keyword for each (fill in `TESTS`).

In [ ]:
TESTS = [
    # (question, keyword that a correct chunk must contain)
    ("Who is solely responsible for AI tool output?", "responsib"),
    # YOUR CODE HERE: add 4 more (question, keyword) pairs from your documents
]

def hit_rate(col, k=4):
    hits = 0
    for q, kw in TESTS:
        if any(kw.lower() in h["text"].lower() for h in retrieve(col, q, k)):
            hits += 1
    return hits / len(TESTS)

print("fixed   :", hit_rate(col_fixed))
if col_sent: print("sentence:", hit_rate(col_sent))

**What to notice:** with only a handful of questions the numbers are noisy. A difference of one question is not conclusive. Try changing `size`, `overlap` and `k` and re-run.

## Stretch goals
1. Add the RBI and HDFC documents and test cross-document questions.
2. Add metadata filtering (`where={"doc": ...}`) to restrict a query to one document.
3. OCR a scanned PDF and add it to the corpus.

## Common errors
- **`ModuleNotFoundError`:** re-run the install cell, then restart the runtime.
- **Connection refused (Ollama):** run `ollama serve` and `ollama pull llama3.1:8b`; use a local notebook, not Colab.
- **Rate limit (Groq free tier):** wait a minute or lower `k`; limits change, check Groq's docs.